# Cartograma de Dorling para Analise de Criminalidade em Belem (PA)
### Pipeline de Processamento Geoespacial e Interpolacao Geometrica em R

Este caderno executa o fluxo computacional em R estruturado a partir da metodologia do repositorio `zumbov2/votemapswitzerland`:
1. Configuracao rapida de dependencias via binarios Linux Posit PPM (sem compilacao demorada de fontes C++).
2. Extracao dinamica de 1 ano de registros criminais diretamente do portal SIAC / CODEC (SEGUP/PA).
3. Reprojecao metrica SIRGAS 2000 UTM 22S (EPSG:31982) e juncao relacional com a malha vetorial de bairros.
4. Simulacao fisica de particulas (particles + tidygraph) para resolucao de sobreposicoes com forcas de atracao e colisao.
5. Interpolacao geometrica continua (transformr + gganimate) para renderizacao da animacao final.

## 1. Instalacao Otimizada de Dependencias (Binarios Posit PPM)

In [ ]:
# 1. Sincronizacao automatica do repositorio no ambiente Colab
if (!dir.exists("R") || !dir.exists("scripts")) {
  message(">> Ambiente Colab detectado: clonando estrutura do repositorio...")
  system("git clone --depth 1 https://github.com/samuel-c-santos/belem-crime-dorling-cartogram.git /tmp/repo")
  system("cp -r /tmp/repo/* .")
  system("rm -rf /tmp/repo")
  message(">> Estrutura sincronizada: pastas R/, scripts/ e data/ estao disponiveis.")
}

# 2. Configuracao do repositorio Posit Public Package Manager (PPM) para Ubuntu 22.04 Jammy
# Baixa binarios Linux pre-compilados em segundos, evitando a compilacao lenta de fontes C++
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

# Dependencias do sistema e do script de extracao Python
system("apt-get update -qq && apt-get install -y -qq libudunits2-dev libgdal-dev libgeos-dev libproj-dev cargo python3-pip", intern = TRUE)
system("pip3 install -q requests pandas openpyxl", intern = TRUE)

# Pacotes R analiticos e graficos
pacotes <- c(
  "sf", "tidyverse", "tidygraph", "particles", 
  "transformr", "gganimate", "gifski", "viridis", "scales", "stringi"
)

novos_pacotes <- pacotes[!(pacotes %in% installed.packages()[, "Package"])]
if (length(novos_pacotes) > 0) {
  install.packages(novos_pacotes, quiet = TRUE)
}

message(">> Ambiente e dependencias configurados com sucesso via binarios pre-compilados.")

## 2. Extracao Dinamica de Dados no SIAC / SEGUP-PA

In [ ]:
# Janela deslizante automatica: data atual menos 365 dias (D-365 ate Hoje)
data_fim    <- as.character(Sys.Date())
data_inicio <- as.character(Sys.Date() - 365)

message(sprintf(">> Disparando requisicao HTTP ao SIAC/CODEC para janela deslizante de 1 ano (%s ate %s)...", data_inicio, data_fim))
cmd <- sprintf("python3 scripts/01_extrair_siac.py %s %s data", data_inicio, data_fim)
system(cmd)
message(">> Extracao concluida e arquivos tabulares gerados em data/processed/.")

## 3. Carga dos Modulos R e Tratamento Metrico

In [ ]:
source("R/01_geometrias_bairros.R")
source("R/02_dorling_fisica.R")
source("R/03_animacao_morphing.R")

bairros_dados <- preparar_geometrias_bairros(
  geojson_path = "data/raw/bairros_belem.geojson",
  crimes_csv_path = "data/processed/crimes_bairros_agregado.csv",
  crs_metrico = 31982
)

# Exibir os 5 bairros com maior volume de ocorrencias registradas
bairros_dados %>%
  st_drop_geometry() %>%
  select(BAIRRO_NORM, TOTAL_CRIMES, ROUBO, FURTO, HOMICIDIO, area_km2) %>%
  arrange(desc(TOTAL_CRIMES)) %>%
  head(5)

## 4. Simulacao Fisica de Particulas (Dorling Layout)

In [ ]:
bairros_dorling <- calcular_dorling_layout(
  bairros_sf = bairros_dados,
  coluna_metrica = "TOTAL_CRIMES",
  fator_escala_k = 40,  # Fator de escala ajustado para volume anual de ~50 mil crimes
  padding = 40,         # Distancia minima de resguardo entre bordas de circulos
  iteracoes = 200
)

message(">> Simulacao de forcas concluida e posicoes de Dorling estabilizadas.")

## 5. Renderizacao da Animacao (Morphing Geometrico)

In [ ]:
output_gif <- gerar_animacao_cartograma(
  bairros_dorling_sf = bairros_dorling,
  coluna_valor = "TOTAL_CRIMES",
  output_file = "outputs/animacao_dorling_seguranca_belem.gif",
  n_frames = 100,
  fps = 18,
  largura = 1100,
  altura = 850
)

# Exibicao do artefato gerado
knitr::include_graphics(output_gif)